# Setup Agentic Information Retrieval
Questo notebook è configurato per l'esecuzione su Google Colab. Assicurati di impostare l'acceleratore hardware su **T4 GPU** (Runtime -> Change runtime type).

In [ ]:
# 1. Verifica che la GPU sia attiva
!nvidia-smi

# 2. Installa le dipendenze core
!pip install -q langchain langgraph langchain-community huggingface-hub pydantic duckduckgo-search ddgs

# 3. Installa llama-cpp-python compilato con supporto CUDA per sfruttare la GPU di Colab
# Rileviamo la versione di CUDA e usiamo la libreria PRE-COMPILATA per evitare 10 minuti di build
wheel_url = "https://abetlen.github.io/llama-cpp-python/whl/cu124"
!pip install llama-cpp-python --extra-index-url {wheel_url} -q


## Clonazione del Codice Sorgente
Poiché hai caricato questo notebook da GitHub, i file Python necessari (`graph.py`, `Refiner.py`, ecc.) non sono automaticamente nel file system di Colab. Dobbiamo clonare la repository per importarli.

In [ ]:
import os

# INSERISCI QUI IL LINK ALLA TUA REPOSITORY:
REPO_URL = "https://github.com/GabCovone/AgenticAskMe"

repo_name = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    !git clone {REPO_URL}

# Spostiamoci nella cartella del progetto per permettere gli import Python
os.chdir(repo_name)
print(f"Directory di lavoro attuale: {os.getcwd()}")

## Inizializzazione del Grafo e del Modello
Grazie alla centralizzazione in `graph.py`, il modello da 14B parametri (circa 10GB) verrà scaricato e caricato in VRAM una sola volta.

In [ ]:
!git pull
from graph import build_graph

# Inizializza il grafo e scarica i pesi del LLM
app = build_graph()
print("Grafo compilato e pronto all'uso!")

## Test con ComplexWebQuestions (CWQ)
Ora testiamo il sistema su domande casuali estratte dal dataset **ComplexWebQuestions** caricato direttamente da HuggingFace.

In [ ]:
from datasets import load_dataset
import random

# Carichiamo il dataset ComplexWebQuestions ufficiale da HuggingFace
print("Scaricamento del dataset CWQ in corso...")
dataset = load_dataset("complex_web_questions", split="train")

# Prendiamo una domanda a caso dal dataset
random_idx = random.randint(0, len(dataset) - 1)
sample = dataset[random_idx]
complex_question = sample['question']
gold_answers = [ans['answer'] for ans in sample['answers']]

print(f"\nDomanda complessa: {complex_question}")
print(f"Risposte corrette (Gold): {gold_answers}\n")

initial_state_cwq = {
    "original_query": complex_question,
    "current_query": complex_question,
    "retrieved_context": "",
    "num_refinement": 0,
    "num_planning": 0,
    "next_node": "",
    "feedback_history": []
}

print("--- AVVIO ESECUZIONE CWQ ---\n")
for output in app.stream(initial_state_cwq):
    for key, value in output.items():
        print(f"---> OUTPUT DAL NODO: {key.upper()} <---")
        print(value)
    print("\n" + "="*50 + "\n")